# Check a notebook before you share it
Run this in your normal notebook environment. Install once in a cell:

```python
%pip install "notebookproof[verify]"
```

This example uses pandas from your existing analysis environment. The base package has no required runtime dependencies. You can use `check(path)` and `frame(df)` without the verification extra.

The example creates only temporary sample files. Fresh verification executes saved code and its normal side effects; run it only on notebooks you intend to execute.

In [ ]:
import notebookproof
import pandas as pd

print("notebookproof", notebookproof.__version__)
df = pd.DataFrame({
    "account_id": [1, 1, 2, 3],
    "amount": pd.Series([10, "10", None, 40], dtype=object),
    "status": ["NA", "NA", "?", None],
})
report = notebookproof.frame(df, keys=["account_id"], rules={"amount": {"nullable": False, "min": 0}})
assert report["rows"] == 4
assert report["columns"][2]["missing"] == 1
assert report["duplicateRows"] == 0
assert report["rulesPassed"] is False
assert next(f for f in report["findings"] if f["code"] == "RULE_TYPE")["rowPositions"] == [2]

## Inspect deliberately broken saved code
The next cell creates a temporary sample notebook. Its saved code uses a value before the definition and contains an old error output. The checker reads that sample without executing it. Cell numbers include Markdown.

On your own notebook, save first and call `notebookproof.check("your-notebook.ipynb")`. Relative paths use your kernel's working directory.

In [ ]:
import json
import tempfile
from pathlib import Path

sample = {"cells": [
    {"cell_type": "markdown", "source": "# Example"},
    {"cell_type": "code", "source": "print(total)", "execution_count": 3,
     "outputs": [{"output_type": "error", "ename": "NameError", "evalue": "total is not defined"}]},
    {"cell_type": "code", "source": "total = 42", "execution_count": 1, "outputs": []},
]}
with tempfile.TemporaryDirectory() as folder:
    sample_path = Path(folder) / "sample.ipynb"
    sample_path.write_text(json.dumps(sample), encoding="utf-8")
    preflight = notebookproof.check(sample_path)

assert preflight["executed"] is False
assert {"FORWARD_REF", "OUT_OF_ORDER", "SAVED_ERROR"} <= {f["code"] for f in preflight["findings"]}

## A notebook that runs successfully but changes its result
The sample below saves an old result of `1` for code that now prints `2`. A fresh run should succeed and identify that output change. The generated HTML summary is displayed here without sharing any data externally. This demonstration is skipped if the verification extra is absent or this notebook is already being verified, so it cannot recursively launch kernels.

In [ ]:
import importlib.util
import os

if importlib.util.find_spec("nbclient") is None:
    print('Install "notebookproof[verify]" in a notebook cell, then rerun this demonstration.')
elif os.environ.get("NOTEBOOKPROOF_VERIFY_ACTIVE") == "1":
    print("Nested verification demonstration skipped; the outer verification is already running.")
else:
    from IPython.display import HTML, display
    with tempfile.TemporaryDirectory() as folder:
        path = Path(folder) / "changed-result.ipynb"
        path.write_text(json.dumps({
            "nbformat": 4, "nbformat_minor": 5, "metadata": {},
            "cells": [{"id": "changed-result", "cell_type": "code", "metadata": {},
                "source": "print(2)", "execution_count": 1,
                "outputs": [{"output_type": "stream", "name": "stdout", "text": "1\n"}]}],
        }), encoding="utf-8")
        full_report = notebookproof.check(path, run=True)
        assert full_report["execution"]["succeeded"] is True
        assert full_report["execution"]["outputsMatch"] is False
        report_path = notebookproof.save_report(full_report, Path(folder) / "report.html")
        display(HTML(report_path.read_text(encoding="utf-8")))

## Try it on your own notebook
Save first. Put the checker in its own cell, beginning with the comment below:

```python
# notebookproof-ignore-output
import notebookproof
report = notebookproof.check("your-notebook.ipynb", run=True)
notebookproof.save_report(report, "analysis-report.html")
```

The comment excludes only this diagnostic cell's output from comparison, and the gap is reported. The code still executes. Notebookproof skips nested verification and its own HTML exports during a fresh run; other notebook code runs normally. The original notebook is never written.

`matched` means supported saved payloads match. `changed` asks you to investigate the difference. `no_baseline`, `partial`, `unsupported`, `ignored`, and unexecuted cells remain visible. This is evidence from one environment; it does not establish analytical correctness.

Existing HTML reports are preserved by default. Choose a new report name or deliberately use `overwrite=True`.

## Tell us what was useful
Did this catch anything you missed? Which warnings were distracting? Could you locate affected table rows? Would you use it again before sharing a notebook? Recruiting for the five-person trial will happen later.